# Object Detection Data Preprocessing Pipeline

This notebook implements a comprehensive data preprocessing pipeline for military equipment object detection. 
The goal is to clean the dataset, handle class imbalance, and prepare it for YOLO training.

## Pipeline Steps:
1. **Audit & Data Cleaning**: Remove noise ('-' class), handle empty images, validate bounding boxes.
2. **Class Remapping**: Merge 'Shishiga' into 'truck', remove 'garbage'.
3. **Stratified Split**: Split data into Train/Val/Test (70/20/10) preserving rare classes.
4. **Formatting & Normalization**: Resize images to 640x640 with letterboxing.
5. **Balanced Training Prep**: Calculate class weights for loss function.

---

In [1]:
pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:

import os
import shutil
import yaml
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib as matplotlib
import seaborn as sns
import mplcyberpunk

# Configuration
DATA_DIR = Path('data')
BACKUP_DIR = Path('data_backup')
IMG_SIZE = 640
KEEP_BACKGROUND_RATIO = 0.1
RANDOM_SEED = 42
matplotlib.style.use("cyberpunk")


# Current Class Map (from data.yaml)
CLASSES = ['-', 'BBM', 'BMP', 'BTR', 'CAY', 'MTLB', 'Shishiga', 'buhanka', 'soldier', 'tank', 'truck']
CLASS_MAP = {i: name for i, name in enumerate(CLASSES)}
print("Current Classes:", CLASS_MAP)

Current Classes: {0: '-', 1: 'BBM', 2: 'BMP', 3: 'BTR', 4: 'CAY', 5: 'MTLB', 6: 'Shishiga', 7: 'buhanka', 8: 'soldier', 9: 'tank', 10: 'truck'}


## 0. Backup Data
Before making any changes, we create a backup of the original data.

In [3]:
if not BACKUP_DIR.exists():
    print(f"Creating backup at {BACKUP_DIR}...")
    shutil.copytree(DATA_DIR, BACKUP_DIR)
    print("Backup created.")
else:
    print("Backup already exists. Skipping.")

Backup already exists. Skipping.


## 1. Audit & Data Cleaning

### 1.1 Collect all files
We gather all image and label files from the existing splits into a single list to process them uniformly.

In [4]:
def get_all_samples(data_dir):
    samples = []
    # Walk through all subdirectories
    for root, dirs, files in os.walk(data_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = Path(root) / file
                # Assume label has same name but .txt extension
                # Check if label exists in 'labels' folder parallel to 'images' or same folder
                # Adjust logic based on actual structure. Assuming standard structure or flat.
                # Trying to find corresponding txt file.
                
                # Strategy: look for .txt in the same directory or in a parallel 'labels' directory
                label_path = img_path.with_suffix('.txt')
                
                # If not in same dir, try replacing 'images' with 'labels' in path
                if not label_path.exists():
                    parts = list(img_path.parts)
                    if 'images' in parts:
                        idx = parts.index('images')
                        parts[idx] = 'labels'
                        label_path_alt = Path(*parts).with_suffix('.txt')
                        if label_path_alt.exists():
                            label_path = label_path_alt
                
                samples.append({'image': img_path, 'label': label_path if label_path.exists() else None})
    return samples

samples = get_all_samples(DATA_DIR)
print(f"Total samples found: {len(samples)}")

Total samples found: 6382


### 1.2 Clean Annotations
- **Remove class '-' (id 0)**.
- **Validate Bounding Boxes**: Clip coordinates to [0, 1]. Remove invalid boxes (w<=0 or h<=0).
- **Identify Background Images**: Images with no valid annotations.

In [5]:
def clean_annotations(samples):
    cleaned_samples = []
    background_samples = []
    stats = {'removed_class_0': 0, 'clipped_boxes': 0, 'invalid_boxes': 0}

    for sample in tqdm(samples, desc="Cleaning annotations"):
        img_path = sample['image']
        label_path = sample['label']
        
        valid_lines = []
        has_changes = False
        
        if label_path and label_path.exists():
            with open(label_path, 'r') as f:
                lines = f.readlines()
            
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                
                cls_id = int(parts[0])
                x, y, w, h = map(float, parts[1:5])
                
                # 1. Remove class '-' (id 0)
                if cls_id == 0:
                    stats['removed_class_0'] += 1
                    has_changes = True
                    continue
                
                # 2. Validate & Clip Coordinates
                # YOLO format: center_x, center_y, width, height (normalized 0-1)
                x_min = x - w / 2
                y_min = y - h / 2
                x_max = x + w / 2
                y_max = y + h / 2
                
                # Clip to [0, 1]
                x_min_c = max(0, min(1, x_min))
                y_min_c = max(0, min(1, y_min))
                x_max_c = max(0, min(1, x_max))
                y_max_c = max(0, min(1, y_max))
                
                if (x_min != x_min_c) or (x_max != x_max_c) or (y_min != y_min_c) or (y_max != y_max_c):
                    stats['clipped_boxes'] += 1
                    has_changes = True
                
                # Recalculate YOLO format
                new_w = x_max_c - x_min_c
                new_h = y_max_c - y_min_c
                new_x = x_min_c + new_w / 2
                new_y = y_min_c + new_h / 2
                
                if new_w <= 0 or new_h <= 0:
                    stats['invalid_boxes'] += 1
                    has_changes = True
                    continue
                
                valid_lines.append(f"{cls_id} {new_x:.6f} {new_y:.6f} {new_w:.6f} {new_h:.6f}\n")
            
            # Save changes if needed
            if has_changes:
                with open(label_path, 'w') as f:
                    f.writelines(valid_lines)
        
        if not valid_lines:
            background_samples.append(sample)
        else:
            cleaned_samples.append(sample)
            
    return cleaned_samples, background_samples, stats

cleaned_samples, background_samples, stats = clean_annotations(samples)
print("Cleaning Stats:", stats)
print(f"Valid samples: {len(cleaned_samples)}")
print(f"Background samples: {len(background_samples)}")

Cleaning annotations:   0%|          | 0/6382 [00:00<?, ?it/s]

Cleaning Stats: {'removed_class_0': 9, 'clipped_boxes': 0, 'invalid_boxes': 0}
Valid samples: 6372
Background samples: 10


### 1.3 Handle Background Images
We keep only a small percentage (10%) of background images to help the model learn negative samples. The rest are deleted.

In [6]:
import random
random.seed(RANDOM_SEED)

num_keep_bg = int(len(cleaned_samples) * KEEP_BACKGROUND_RATIO)
num_keep_bg = min(num_keep_bg, len(background_samples))

keep_bg = random.sample(background_samples, num_keep_bg)
delete_bg = [s for s in background_samples if s not in keep_bg]

print(f"Keeping {len(keep_bg)} background images. Deleting {len(delete_bg)}.")

# Delete files
for s in delete_bg:
    try:
        os.remove(s['image'])
        if s['label'] and s['label'].exists():
            os.remove(s['label'])
    except Exception as e:
        print(f"Error deleting {s['image']}: {e}")

# Add kept background samples back to main list
final_samples = cleaned_samples + keep_bg

Keeping 10 background images. Deleting 0.


## 2. Class Remapping

**Logic**:
- `Shishiga` (id 6) -> `truck` (id 10)
- `BBM` (id 1) -> Keep for now (or merge if requested)
- `tank` (id 9) -> Keep
- `soldier` (id 8) -> Keep
- `-` (id 0) -> Already removed

We will update the class IDs in the annotation files.

In [7]:
def remap_classes(samples):
    # Define mapping: Old ID -> New ID
    # 6 (Shishiga) -> 10 (truck)
    mapping = {
        6: 10
    }
    
    remapped_count = 0
    
    for sample in tqdm(samples, desc="Remapping classes"):
        label_path = sample['label']
        if not label_path or not label_path.exists():
            continue
            
        with open(label_path, 'r') as f:
            lines = f.readlines()
        
        new_lines = []
        file_changed = False
        
        for line in lines:
            parts = line.strip().split()
            cls_id = int(parts[0])
            
            if cls_id in mapping:
                parts[0] = str(mapping[cls_id])
                new_lines.append(" ".join(parts) + "\n")
                file_changed = True
                remapped_count += 1
            else:
                new_lines.append(line)
        
        if file_changed:
            with open(label_path, 'w') as f:
                f.writelines(new_lines)
                
    print(f"Remapped {remapped_count} annotations.")

remap_classes(final_samples)

Remapping classes:   0%|          | 0/6382 [00:00<?, ?it/s]

Remapped 4 annotations.


## 3. Stratified Split

We use **Stratified Sampling** based on the rarest class in each image to ensure all classes are represented in Validation and Test sets.

In [8]:
def get_rarest_class(label_path):
    if not label_path or not label_path.exists():
        return -1 # Background
    
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    classes = [int(line.split()[0]) for line in lines]
    if not classes:
        return -1
    
    # Here we should ideally know global class counts to find the rarest.
    # For simplicity, we can just return the max class ID or min, but better to use frequency.
    # Let's count global frequencies first.
    return classes # Return all classes to analyze later

# 1. Count global class frequencies
global_counts = {}
sample_classes = []

for sample in final_samples:
    cls_list = get_rarest_class(sample['label'])
    if cls_list == -1:
        sample_classes.append([-1])
    else:
        sample_classes.append(cls_list)
        for c in cls_list:
            global_counts[c] = global_counts.get(c, 0) + 1

print("Global Class Counts:", global_counts)

# 2. Assign 'stratify_label' to each image (rarest class present)
stratify_labels = []
for cls_list in sample_classes:
    if cls_list == [-1]:
        stratify_labels.append(-1)
    else:
        # Find class with min count
        rarest = min(cls_list, key=lambda x: global_counts.get(x, float('inf')))
        stratify_labels.append(rarest)

# 3. Split
train_samples, val_test_samples, train_labels, val_test_labels = train_test_split(
    final_samples, stratify_labels, test_size=0.3, stratify=stratify_labels, random_state=RANDOM_SEED
)

val_samples, test_samples, val_labels, test_labels = train_test_split(
    val_test_samples, val_test_labels, test_size=0.33, stratify=val_test_labels, random_state=RANDOM_SEED
)

print(f"Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}")

Global Class Counts: {9: 3649, 10: 496, 2: 589, 3: 648, 4: 106, 8: 2723, 7: 92, 5: 95, 1: 49}
Train: 4467, Val: 1283, Test: 632


### 3.1 Move Files to New Structure
We reorganize the files into `train`, `valid`, `test` folders.

In [9]:
def move_files(samples, split_name):
    dest_img_dir = DATA_DIR / split_name / 'images'
    dest_lbl_dir = DATA_DIR / split_name / 'labels'
    
    dest_img_dir.mkdir(parents=True, exist_ok=True)
    dest_lbl_dir.mkdir(parents=True, exist_ok=True)
    
    for sample in tqdm(samples, desc=f"Moving to {split_name}"):
        src_img = sample['image']
        src_lbl = sample['label']
        
        # Move Image
        shutil.move(str(src_img), str(dest_img_dir / src_img.name))
        
        # Move Label
        if src_lbl and src_lbl.exists():
            shutil.move(str(src_lbl), str(dest_lbl_dir / src_lbl.name))

# Clear existing dirs first to avoid duplicates if re-running (optional, be careful)
# For now, we assume we are moving from the scattered structure to this new one.
# Since we are moving files, the original locations will be empty.

move_files(train_samples, 'train')
move_files(val_samples, 'valid')
move_files(test_samples, 'test')

Moving to train:   0%|          | 0/4467 [00:00<?, ?it/s]

Moving to valid:   0%|          | 0/1283 [00:00<?, ?it/s]

Moving to test:   0%|          | 0/632 [00:00<?, ?it/s]

## 4. Formatting & Normalization

**Resize**: Resize all images to 640x640 using Letterboxing (padding) to preserve aspect ratio.

In [10]:
def letterbox_resize(img, target_size=640):
    h, w = img.shape[:2]
    scale = min(target_size / h, target_size / w)
    nw, nh = int(w * scale), int(h * scale)
    
    resized = cv2.resize(img, (nw, nh))
    
    # Create canvas
    canvas = np.full((target_size, target_size, 3), 114, dtype=np.uint8) # 114 is standard YOLO gray
    
    # Center
    x_off = (target_size - nw) // 2
    y_off = (target_size - nh) // 2
    
    canvas[y_off:y_off+nh, x_off:x_off+nw] = resized
    return canvas

def process_images_resize(data_dir):
    image_files = list(data_dir.rglob('*.jpg')) + list(data_dir.rglob('*.png'))
    
    for img_path in tqdm(image_files, desc="Resizing images"):
        img = cv2.imread(str(img_path))
        if img is None: continue
        
        # Check if resize needed
        h, w = img.shape[:2]
        if h == IMG_SIZE and w == IMG_SIZE:
            continue
            
        new_img = letterbox_resize(img, IMG_SIZE)
        cv2.imwrite(str(img_path), new_img)

process_images_resize(DATA_DIR)

Resizing images:   0%|          | 0/6382 [00:00<?, ?it/s]

## 5. Balanced Training Prep (Metadata)

Calculate class weights for the loss function.

In [11]:
# Recalculate counts after split and cleaning
final_counts = {}
total_samples = 0

for split in ['train', 'valid', 'test']:
    lbl_dir = DATA_DIR / split / 'labels'
    for lbl_file in lbl_dir.glob('*.txt'):
        with open(lbl_file, 'r') as f:
            for line in f:
                c = int(line.split()[0])
                final_counts[c] = final_counts.get(c, 0) + 1
                total_samples += 1

print("Final Class Counts:", final_counts)

# Calculate Weights: Total / (NumClasses * ClassCount)
num_classes = len(final_counts)
class_weights = {}

for c, count in final_counts.items():
    weight = total_samples / (num_classes * count)
    class_weights[c] = round(weight, 4)

print("Class Weights:", class_weights)

# Save to yaml
with open(DATA_DIR / 'class_weights.yaml', 'w') as f:
    yaml.dump(class_weights, f)
    
print("Saved class weights to class_weights.yaml")

Final Class Counts: {4: 106, 9: 3649, 8: 2723, 3: 648, 2: 589, 5: 95, 10: 496, 1: 49, 7: 92}
Class Weights: {4: 8.8543, 9: 0.2572, 8: 0.3447, 3: 1.4484, 2: 1.5935, 5: 9.8795, 10: 1.8922, 1: 19.1542, 7: 10.2017}
Saved class weights to class_weights.yaml


## Conclusion

The data has been cleaned, remapped, stratified, and resized. 
Class weights have been calculated to handle imbalance.
The dataset is now ready for training.